# 05 — Sentiment Analysis

**Project:** Customer Complaint Root-Cause Classifier  
**Module:** 4 — Complaint Narrative Sentiment Analysis  
**Target:** `sentiment_label` (`Negative`, `Neutral`, `Positive`)  
**Input:** `complaint_narrative`  

### Objectives:
1. Load complaints dataset and analyze sentiment class distribution and imbalance (64.1% Negative, 24.0% Neutral, 11.9% Positive).
2. Apply **Group-Aware Splitting** (`GroupShuffleSplit` on narrative MD5 hash) to eliminate synthetic duplicate text leakage.
3. Extract sparse n-gram features via **TF-IDF Vectorizer** (bounded memory under 8 GB RAM).
4. Train and benchmark 3 candidate models:
   - **Logistic Regression** (with `class_weight='balanced'`)
   - **Linear Support Vector Classifier** (with `class_weight='balanced'`)
   - **Multinomial Naive Bayes**
5. Evaluate using **Macro F1** (critical for imbalanced classes), Weighted F1, Accuracy, and Confusion Matrices.
6. Save the champion sentiment model artifacts to `models/sentiment_classifier.joblib`.
7. Validate and compare against pretrained transformer model (`cardiffnlp/twitter-roberta-base-sentiment-latest`).

## 1. Setup and Environment

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path(os.path.abspath("")).resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

from config.settings import (
    PROJECT_ROOT, MODELS_DIR, RANDOM_SEED, TEST_SIZE, TFIDF_MAX_FEATURES
)
from config.labels import SENTIMENT_LABELS
from src.data.load_data import load_complaints
from src.data.clean_data import clean_complaints
from src.utils.helpers import generate_narrative_hash, get_memory_usage
from src.classification.evaluate import evaluate_model, plot_confusion_matrix, compare_models

print(f"Project root: {PROJECT_ROOT}")
print(f"Memory usage: {get_memory_usage():.2f} MB")

## 2. Load Data and Inspect Sentiment Distribution

In [ ]:
print("Loading dataset...")
raw_df = load_complaints()
df = clean_complaints(raw_df)

sentiment_counts = df["sentiment_label"].value_counts()
sentiment_pct = df["sentiment_label"].value_counts(normalize=True) * 100

print("Sentiment Distribution:")
for label, count in sentiment_counts.items():
    print(f"  {label:10s}: {count:,} ({sentiment_pct[label]:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {"Negative": "#e74c3c", "Neutral": "#f39c12", "Positive": "#27ae60"}
bar_colors = [colors[l] for l in sentiment_counts.index]

axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=bar_colors, edgecolor="black", alpha=0.85)
axes[0].set_title("Sentiment Count (Imbalance)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Complaints")
for i, v in enumerate(sentiment_counts.values):
    axes[0].text(i, v + 800, f"{v:,}", ha="center", fontweight="bold")

axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct="%1.1f%%", colors=bar_colors, startangle=140)
axes[1].set_title("Sentiment Proportions", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Group-Aware Split (Prevent Data Leakage)

In [ ]:
X = df["complaint_narrative"]
y_raw = df["sentiment_label"]

# MD5 group hashes to keep duplicate narratives in the same fold
groups = X.apply(generate_narrative_hash)

# Encode categorical target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
classes = label_encoder.classes_

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train size: {len(X_train):,} ({len(X_train)/len(df):.1%})")
print(f"Test size:  {len(X_test):,} ({len(X_test)/len(df):.1%})")

# Verify 0 leakage
overlap = set(groups.iloc[train_idx]).intersection(set(groups.iloc[test_idx]))
assert len(overlap) == 0, "Group leakage detected!"
print("Group isolation verified: Zero overlap between train and test narratives.")

## 4. Sparse Feature Extraction (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.95,
)

print("Fitting TF-IDF on narrative texts...")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"X_train shape: {X_train_vec.shape}")
print(f"X_test shape:  {X_test_vec.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")

## 5. Model Training & Comparison

> **Class imbalance handling**: We use `class_weight='balanced'` in Logistic Regression and LinearSVC to penalize under-predictions on the minority Positive class (11.9%).

In [ ]:
candidate_models = {
    "Logistic Regression (Balanced)": LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
    "Linear SVM (Balanced)": LinearSVC(
        max_iter=2000,
        class_weight="balanced",
        dual="auto",
        random_state=RANDOM_SEED,
    ),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
}

evaluations = []
trained_models = {}
predictions = {}

for name, model in candidate_models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train_vec, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_vec)
    predictions[name] = y_pred
    
    metrics = evaluate_model(y_test, y_pred, labels=classes, model_name=name)
    evaluations.append(metrics)
    print(f"{name} -> Accuracy: {metrics['accuracy']:.4f} | Macro F1: {metrics['macro_f1']:.4f} | Weighted F1: {metrics['weighted_f1']:.4f}")

## 6. Sentiment Benchmark Comparison

In [ ]:
comparison_df = compare_models(evaluations)
print("=== SENTIMENT CLASSIFICATION BENCHMARK ===")
print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"Champion Sentiment Classifier: {best_model_name}")

# Plot normalized confusion matrix
fig = plot_confusion_matrix(
    y_test,
    predictions[best_model_name],
    labels=classes,
    model_name=best_model_name,
    figsize=(8, 6)
)
plt.title(f"Confusion Matrix: {best_model_name}", fontsize=12, fontweight="bold")
plt.show()

## 7. Save Champion Sentiment Artifacts

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

sent_model_file = MODELS_DIR / "sentiment_classifier.joblib"
sent_vec_file = MODELS_DIR / "tfidf_sentiment_vectorizer.joblib"
sent_le_file = MODELS_DIR / "sentiment_label_encoder.joblib"

joblib.dump(best_model, sent_model_file)
joblib.dump(vectorizer, sent_vec_file)
joblib.dump(label_encoder, sent_le_file)

print(f"Saved sentiment model -> {sent_model_file}")
print(f"Saved TF-IDF vectorizer -> {sent_vec_file}")
print(f"Saved label encoder    -> {sent_le_file}")

## 8. Pretrained Transformer Comparison & Domain Discussion

We contrast our local TF-IDF model against **`cardiffnlp/twitter-roberta-base-sentiment-latest`**.

> **Key Domain Insight**: Financial complaints differ markedly from general social media text. Customers express extreme dissatisfaction through formal institutional language (*"I am requesting an immediate investigation and correction"*, *"This error violates my account agreement"*) rather than explicit casual sentiment words (*"hate"*, *"terrible"*). Pretrained social media models can mistake polite/formal language for Neutral.

In [ ]:
from src.sentiment.sentiment_analyzer import predict_sentiment, predict_sentiment_roberta

sample_complaints = [
    "I was charged twice for one transaction. This situation has been frustrating and needs attention.",
    "I am requesting clarification regarding my statement before taking any further action.",
    "Thank you for resolving the billing dispute quickly. The refund appeared on my account as promised.",
    "My debit card was locked without any warning and I was unable to buy groceries for my family."
]

print("=== SENTIMENT PREDICTION COMPARISON ===\n")
for text in sample_complaints:
    local_res = predict_sentiment(text)
    print(f"Narrative: {text[:80]}...")
    print(f"  [TF-IDF Champion] Sentiment: {local_res['predicted_sentiment']:8s} | Conf: {local_res['confidence']:.2%} | Score: {local_res['sentiment_score']:+.2f}")
    
    try:
        roberta_res = predict_sentiment_roberta(text)
        print(f"  [RoBERTa Model]   Sentiment: {roberta_res['predicted_sentiment']:8s} | Conf: {roberta_res['confidence']:.2%} | Score: {roberta_res['sentiment_score']:+.2f}")
    except Exception as e:
        print(f"  [RoBERTa Model]   Skipped (HuggingFace offline / uninstalled: {e})")
    print("-" * 75)